In [1]:
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

In [2]:
load_dotenv()

True

In [3]:
model = init_chat_model("google_genai:gemini-2.5-flash-lite")

SEQUENTIAL CHAIN

In [4]:
prompt = PromptTemplate(
    template = "Generate 5 interesting facts about {topic}",
    input_variables= ["topic"]
)

In [5]:
parser = StrOutputParser()

In [6]:
chain = prompt | model | parser
result = chain.invoke({"topic": "stars"})
print(result)

Here are 5 interesting facts about stars:

1.  **Stars are born in giant clouds of gas and dust called nebulae.** These nebulae are incredibly vast, spanning light-years across. Gravity within denser regions of the nebula causes the gas and dust to clump together. As these clumps grow, they become hotter and denser, eventually igniting nuclear fusion in their core, marking the birth of a star.

2.  **The Sun is an average-sized star, and our universe is teeming with stars of all sizes, from tiny red dwarfs to colossal supergiants.** Red dwarfs are the most common type of star, being much smaller and cooler than our Sun, and can live for trillions of years. On the other end of the spectrum, supergiants are enormous, hundreds or even thousands of times the Sun's diameter, and burn through their fuel incredibly quickly, ending their lives in spectacular supernova explosions.

3.  **Stars don't twinkle because they are inherently unstable; they twinkle because of Earth's atmosphere.** When

In [7]:
# TO PRINT STEPS OF CHAIN
chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
    +-----------------+    
    | StrOutputParser |    
    +-----------------+    
             *             
             *             
             *             
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  


SEQUENTIAL CHAIN(MULITPLE LLM CALLS)

In [10]:
prompt1 = PromptTemplate(
    template = "Generate detailed report on {topic}",
    input_variables= ["topic"]
)

prompt2 = PromptTemplate(
    template = "Generate 5 pointer summary from the following text \n {text}",
    input_variables= ["text"]
)

chain = prompt1 | model | parser | prompt2 | model | parser

result = chain.invoke({"topic": "Unemployment"})

print(result)

Here is a 5-pointer summary of the provided text:

*   **Definition and Measurement:** Unemployment is defined as individuals actively seeking work but unable to find it, measured by rates, participation, duration, and underemployment.
*   **Global and National Trends:** Unemployment fluctuates with economic cycles globally, showing regional disparities and facing new challenges from technology, globalization, and events like pandemics.
*   **Multifaceted Causes:** Unemployment stems from cyclical downturns, structural issues like skills mismatches and industry shifts, frictional job transitions, seasonal demands, and other factors like regulations and demographic changes.
*   **Significant Impacts:** Unemployment causes individual hardship (economic, psychological, health), strains families, and leads to broader societal issues like reduced economic output, lower tax revenues, increased government spending, poverty, and social unrest.
*   **Comprehensive Policy Responses:** Addressing

In [11]:
chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
    +-----------------+    
    | StrOutputParser |    
    +-----------------+    
             *             
             *             
             *             
+-----------------------+  
| StrOutputParserOutput |  
+-----------------------+  
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *      

PARALLEL CHAINS

In [16]:
from langchain_core.runnables import RunnableParallel

In [17]:
prompt1 = PromptTemplate(
    template = "Generate short and simple notes from the following text \n {text}",
    input_variables= ["text"]
)

prompt2 = PromptTemplate(
    template = "Generate 5 short question answers from the following text \n {text}",
    input_variables= ["text"]
)

prompt3 = PromptTemplate(
    template = "Merge the provided notes and quiz into a single document \n notes -> {notes} and quiz -> {quiz}",
    input_variables= ["notes", "quiz"]
)

parallel_chain = RunnableParallel({
    "notes": prompt1 | model | parser,
    "quiz": prompt2 | model | parser
})

merge_chain = prompt3 | model | parser 

chain = parallel_chain | merge_chain

text = """
Prompt engineering is a relatively new discipline for developing and optimizing prompts to efficiently use language models (LMs) for a wide variety of applications and research topics. Prompt engineering skills help to better understand the capabilities and limitations of large language models (LLMs).

Researchers use prompt engineering to improve the capacity of LLMs on a wide range of common and complex tasks such as question answering and arithmetic reasoning. Developers use prompt engineering to design robust and effective prompting techniques that interface with LLMs and other tools.

Prompt engineering is not just about designing and developing prompts. It encompasses a wide range of skills and techniques that are useful for interacting and developing with LLMs. It's an important skill to interface, build with, and understand capabilities of LLMs. You can use prompt engineering to improve safety of LLMs and build new capabilities like augmenting LLMs with domain knowledge and external tools.

Motivated by the high interest in developing with LLMs, we have created this new prompt engineering guide that contains all the latest papers, advanced prompting techniques, learning guides, model-specific prompting guides, lectures, references, new LLM capabilities, and tools related to prompt engineering.


"""

result = chain.invoke({"text": text})

print(result)

Here's the merged document combining your notes and quiz:

## Prompt Engineering: Notes and Quiz

**Notes:**

*   **Prompt Engineering:** This is a new field focused on making prompts work better with language models (LMs).
*   **Helps:** It assists in understanding the abilities and limits of LMs.
*   **Used by:**
    *   **Researchers:** To improve LM tasks such as Q&A and math problems.
    *   **Developers:** To create effective ways of using LMs with various tools.
*   **More than just prompts:** This field also encompasses skills for interacting with LMs in general.
*   **Benefits:**
    *   Leads to better LM interaction.
    *   Enhances LM safety.
    *   Enables the development of new LM abilities (e.g., incorporating external knowledge or tools).
*   **Guide Created:** A guide has been developed to cover the latest papers, techniques, guides, and tools related to prompt engineering.

---

**Quiz:**

1.  **Q: What is prompt engineering?**
    **A:** It's a discipline for deve

In [18]:
chain.get_graph().print_ascii()

                    +---------------------------+                      
                    | Parallel<notes,quiz>Input |                      
                    +---------------------------+                      
                       ***                   ***                       
                   ****                         ****                   
                 **                                 **                 
    +----------------+                          +----------------+     
    | PromptTemplate |                          | PromptTemplate |     
    +----------------+                          +----------------+     
             *                                           *             
             *                                           *             
             *                                           *             
+------------------------+                  +------------------------+ 
| ChatGoogleGenerativeAI |                  | ChatGoogleGenerati

CONDITIONAL CHAIN

In [19]:
from pydantic import BaseModel, Field
from typing import Literal
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda

In [ ]:
class Feedback(BaseModel):

    sentiment: Literal['positive', 'negative'] = Field(description='Give the sentiment of the feedback')

parser2 = PydanticOutputParser(pydantic_object=Feedback)

prompt1 = PromptTemplate(
    template='Classify the sentiment of the following feedback text into postive or negative \n {feedback} \n {format_instruction}',
    input_variables=['feedback'],
    partial_variables={'format_instruction':parser2.get_format_instructions()}
)

classifier_chain = prompt1 | model | parser2

prompt2 = PromptTemplate(
    template='Write an appropriate response to this positive feedback \n {feedback}',
    input_variables=['feedback']
)

prompt3 = PromptTemplate(
    template='Write an appropriate response to this negative feedback \n {feedback}',
    input_variables=['feedback']
)

branch_chain = RunnableBranch(
    (lambda x:x.sentiment == 'positive', prompt2 | model | parser),
    (lambda x:x.sentiment == 'negative', prompt3 | model | parser),
    RunnableLambda(lambda x: "could not find sentiment")
)

chain = classifier_chain | branch_chain

print(chain.invoke({'feedback': 'This is a beautiful phone'}))

Here are several appropriate responses to positive feedback, categorized by formality and nuance. Choose the one that best fits your relationship with the person giving the feedback and the context:

**Short & Sweet (Good for quick acknowledgments):**

*   "Thank you so much!"
*   "I really appreciate that."
*   "That's great to hear!"
*   "Glad I could help/contribute!"
*   "Thank you for the kind words!"

**Slightly More Detailed (Adds a touch more warmth):**

*   "Thank you, I'm so glad to hear that!"
*   "That means a lot to me, thank you!"
*   "I really appreciate you saying that. Thank you!"
*   "Thank you! I'm happy you found it [helpful/useful/good]."
*   "Thanks! It was a pleasure working on that."

**More Formal (Good for professional settings, superiors, or clients):**

*   "Thank you for your positive feedback. I'm pleased to hear that."
*   "I appreciate your kind words and I'm glad I could meet your expectations."
*   "Thank you for recognizing my efforts. I'm happy that 

In [21]:
chain.get_graph().print_ascii()

      +-------------+      
      | PromptInput |      
      +-------------+      
             *             
             *             
             *             
    +----------------+     
    | PromptTemplate |     
    +----------------+     
             *             
             *             
             *             
+------------------------+ 
| ChatGoogleGenerativeAI | 
+------------------------+ 
             *             
             *             
             *             
 +----------------------+  
 | PydanticOutputParser |  
 +----------------------+  
             *             
             *             
             *             
        +--------+         
        | Branch |         
        +--------+         
             *             
             *             
             *             
     +--------------+      
     | BranchOutput |      
     +--------------+      
